[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-2/lab-2.2-finding-the-spill.ipynb)

# LAB·2.2 · Finding the spill in naive attention

**Hardware:** TPU runtime for the optimized dump and the profile; the estimate works anywhere.

The single observation this whole track is built on: XLA fuses well, but naive attention's score matrix still crosses HBM, because avoiding it needs an algebraic rewrite that no fusion pass can perform. Today you find that spill in the compiler's own output and put a number on it.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
INTERP = not ON_TPU  # interpret mode anywhere; compiled kernels on a real TPU

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


In [ ]:
S, D = 8192, 128

def naive_attention(q, k, v):
    s = q @ k.T
    m = jnp.max(s, axis=-1, keepdims=True)
    p = jnp.exp(s - m)
    l = jnp.sum(p, axis=-1, keepdims=True)
    return (p / l) @ v

args = [jax.ShapeDtypeStruct((S, D), jnp.bfloat16)] * 3

# the spill, predicted from first principles: the SxS score matrix in bf16,
# written once after the first matmul and read back for the softmax chain
score_bytes = S * S * 2
print(f"score matrix: {score_bytes / 1e6:.0f} MB; at least written + read once = {2 * score_bytes / 1e6:.0f} MB of HBM traffic")
print(f"at v5e HBM bandwidth (8.2e11 B/s): {2 * score_bytes / 8.2e11 * 1e3:.1f} ms of pure spill time")

## Hunt it in the dump (TPU runtime)

Print the optimized HLO and search for `bf16[8192,8192]` buffers that appear as fusion *outputs*: each one is a full score matrix round-tripping through HBM. Count them; the multi-pass softmax typically forces more than one.

In [ ]:
if ON_TPU:
    hlo = jax.jit(naive_attention).lower(*args).compile().as_text()
    lines = [ln for ln in hlo.splitlines() if "8192,8192" in ln and ("fusion" in ln or "custom-call" in ln or "= bf16" in ln)]
    print(f"{len(lines)} lines mention the full score matrix; the ones that are fusion outputs are your spills:")
    for ln in lines[:12]:
        print(ln.strip()[:160])
else:
    print("Optimized-HLO hunt needs the TPU runtime; the prediction above works anywhere.")

## Confirm with the profiler

Trace one call under `jax.profiler.trace`, open TensorBoard, and find the HBM traffic of the attention fusions. Gate criterion: your predicted spill bytes within 20% of what the profiler reports. Then write the sentence that Stage 3 exists to fix: *the spill is not a fusion failure; it is an algorithm failure.*

In [ ]:
if ON_TPU:
    fn = jax.jit(naive_attention)
    xs = [jax.random.normal(jax.random.key(i), (S, D), jnp.bfloat16) for i in range(3)]
    fn(*xs).block_until_ready()
    with jax.profiler.trace("/tmp/spill-trace"):
        fn(*xs).block_until_ready()
    print("trace at /tmp/spill-trace; compare measured HBM bytes to the prediction above")